# 08 – Model Scale Sweep

## Hypothesis

Larger encoder capacity provides richer scene representations, which should translate into better set predictions. However, at ~5k training scenes the bottleneck may not be the encoder: overfitting or data scale could dominate, meaning we see diminishing returns – or even degradation – beyond a certain `d_model`.

## Experiment

Fix the loss to **PowerSoftMin³ τ=0.12** and sweep `d_model ∈ {64, 128, 256, 512}`, keeping `latent_dim = hidden_dim = d_model`. All other hyperparameters are held constant.

| Label | d_model | latent_dim | hidden_dim |
|---|---|---|---|
| `d_model=64`  | 64  | 64  | 64  |
| `d_model=128` | 128 | 128 | 128 |
| `d_model=256` | 256 | 256 | 256 |
| `d_model=512` | 512 | 512 | 512 |

Entropy and PCA snapshots captured at epochs 0, 5, 15, 35, 49.

In [ ]:
import sys
sys.path.insert(0, '.')
import clevr_utils as cu
from influencerformer.losses import PowerSoftMinLoss
print(f'Device: {cu.DEVICE}')

In [ ]:
D_MODEL_LIST = [64, 128, 256, 512]
N_EPOCHS = 50
SNAPSHOT_EPOCHS = {0, 5, 15, 35, 49}
TAU = 0.12

In [ ]:
all_results = {}

for d in D_MODEL_LIST:
    name = f'd_model={d}'
    m, s = cu.train_and_monitor(
        PowerSoftMinLoss(temperature=TAU, power=3.0),
        name,
        model_fn=lambda d=d: cu.make_model(d_model=d, latent_dim=d, hidden_dim=d),
        n_epochs=N_EPOCHS,
        snapshot_epochs=SNAPSHOT_EPOCHS,
        tau_for_entropy=TAU,
    )
    all_results[name] = (m, s)

In [ ]:
cu.plot_monitoring(all_results)

# PCA snapshot comparison: d_model=64 vs d_model=256
keys_to_compare = [k for k in ['d_model=64', 'd_model=256'] if k in all_results]
if len(keys_to_compare) == 2:
    cu.plot_pca_snapshots(
        {k: all_results[k] for k in keys_to_compare}
    )
else:
    print(f'PCA comparison skipped – available keys: {list(all_results.keys())}')

In [ ]:
cu.summary_table(all_results)